# VLM + OpenCV Axis Detection Notebook

Deze notebook combineert:

## 1. OpenCV
OpenCV detecteert mogelijke assen met line-support scanning.

## 2. VLM
Een Vision-Language Model wordt gebruikt als extra reasoning/validatie-laag.

De VLM doet dus niet het harde pixelwerk, maar helpt met vragen zoals:

```text
- Is this mainly a horizontal bar chart or vertical bar chart?
- Are the detected axes logical?
- Are some detections probably dividers or borders?
- Are there multiple charts in this image?
```

## Waarom deze aanpak?

Een VLM kan goed redeneren over een afbeelding, maar is minder geschikt voor pixel-perfect lijncoördinaten.  
OpenCV kan juist goed met pixels en lijnen werken, maar begrijpt geen grafiekcontext.

Daarom:

```text
OpenCV = exacte lijn-kandidaten
VLM = context en validatie
```


In [ ]:
# Install packages
%pip install opencv-python matplotlib numpy pandas pillow

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import base64
import json
import os

print("OpenCV version:", cv2.__version__)

## 1. Settings

Pas `IMAGE_PATH` aan naar jouw afbeelding.

Voorbeeld:

```python
IMAGE_PATH = Path("../Dataset/Compliant/chart1.png")
```


In [ ]:
IMAGE_PATH = Path("../Dataset/Compliant/example.png")

OUTPUT_DIR = Path("../output/vlm_opencv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# OpenCV settings
DARK_THRESHOLD = 235
SATURATION_MAX = 190
BORDER_MARGIN_RATIO = 0.025

MIN_VERTICAL_LINE_HEIGHT = 25
MAX_VERTICAL_LINE_WIDTH = 10
MIN_HORIZONTAL_LINE_WIDTH = 25
MAX_HORIZONTAL_LINE_HEIGHT = 10
MAX_LINE_SPAN_RATIO = 0.80

RIGHT_SCAN_GAP = 8
RIGHT_SCAN_WIDTH = 180
MIN_RIGHT_RUN_PIXELS = 18
MIN_BAR_BAND_HEIGHT = 3

UP_SCAN_GAP = 8
UP_SCAN_HEIGHT = 160
MIN_UP_RUN_PIXELS = 18
MIN_BAR_BAND_WIDTH = 3

MIN_SUPPORTING_BARS = 2
MAX_AXES_TO_DRAW = 30

## 2. Helper functions

In [ ]:

def show_image(title, image, figsize=(12, 8), cmap=None):
    plt.figure(figsize=figsize)

    if len(image.shape) == 2:
        plt.imshow(image, cmap=cmap or "gray")
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    plt.title(title)
    plt.axis("off")
    plt.show()


def load_image(path):
    image = cv2.imread(str(path))

    if image is None:
        raise FileNotFoundError(f"Image not found: {path}")

    return image


def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


# Part A - OpenCV axis detection

Deze OpenCV-code gebruikt de line-support scanning aanpak:

## Horizontal bar chart

```text
vertical line candidate
↓
scan right side
↓
multiple horizontal bar segments found
↓
left_axis
```

## Vertical bar chart

```text
horizontal line candidate
↓
scan above
↓
multiple vertical bar segments found
↓
bottom_x_axis
```


In [ ]:

def create_dark_mask(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]

    mask = np.zeros_like(gray, dtype=np.uint8)
    mask[(gray < DARK_THRESHOLD) & (saturation < SATURATION_MAX)] = 255

    return mask


def remove_outer_border(mask):
    h, w = mask.shape[:2]
    mx = int(w * BORDER_MARGIN_RATIO)
    my = int(h * BORDER_MARGIN_RATIO)

    cleaned = mask.copy()
    cleaned[:my, :] = 0
    cleaned[h-my:, :] = 0
    cleaned[:, :mx] = 0
    cleaned[:, w-mx:] = 0

    return cleaned


def make_horizontal_structure_mask(mask):
    h, w = mask.shape[:2]
    kernel_width = max(15, int(w * 0.025))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_width, 3))

    horizontal_mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return horizontal_mask


def make_vertical_structure_mask(mask):
    h, w = mask.shape[:2]
    kernel_height = max(15, int(h * 0.025))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, kernel_height))

    vertical_mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return vertical_mask


def detect_vertical_line_candidates(mask, image_shape):
    h, w = image_shape[:2]
    kernel_height = max(20, int(h * 0.035))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_height))

    vertical_lines_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    contours, _ = cv2.findContours(vertical_lines_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    lines = []

    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)

        if bh < MIN_VERTICAL_LINE_HEIGHT:
            continue

        if bw > MAX_VERTICAL_LINE_WIDTH:
            continue

        if bh > h * MAX_LINE_SPAN_RATIO:
            continue

        lines.append({
            "axis_type": "left_axis",
            "x1": int(x + bw / 2),
            "y1": int(y),
            "x2": int(x + bw / 2),
            "y2": int(y + bh),
            "width": int(bw),
            "height": int(bh)
        })

    return lines, vertical_lines_mask


def detect_horizontal_line_candidates(mask, image_shape):
    h, w = image_shape[:2]
    kernel_width = max(20, int(w * 0.035))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_width, 1))

    horizontal_lines_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    contours, _ = cv2.findContours(horizontal_lines_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    lines = []

    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)

        if bw < MIN_HORIZONTAL_LINE_WIDTH:
            continue

        if bh > MAX_HORIZONTAL_LINE_HEIGHT:
            continue

        if bw > w * MAX_LINE_SPAN_RATIO:
            continue

        lines.append({
            "axis_type": "bottom_x_axis",
            "x1": int(x),
            "y1": int(y + bh / 2),
            "x2": int(x + bw),
            "y2": int(y + bh / 2),
            "width": int(bw),
            "height": int(bh)
        })

    return lines, horizontal_lines_mask


def count_horizontal_bar_support(line, horizontal_mask):
    h, w = horizontal_mask.shape[:2]
    x = line["x1"]
    y1 = max(0, line["y1"])
    y2 = min(h - 1, line["y2"])

    scan_x1 = min(w - 1, x + 1)
    scan_x2 = min(w, x + RIGHT_SCAN_GAP + RIGHT_SCAN_WIDTH)

    if scan_x2 <= scan_x1 or y2 <= y1:
        return 0, []

    region = horizontal_mask[y1:y2 + 1, scan_x1:scan_x2]
    row_counts = np.count_nonzero(region, axis=1)
    supported_rows = row_counts >= MIN_RIGHT_RUN_PIXELS

    bands = []
    start = None

    for idx, value in enumerate(supported_rows):
        if value and start is None:
            start = idx
        elif not value and start is not None:
            end = idx - 1
            if end - start + 1 >= MIN_BAR_BAND_HEIGHT:
                bands.append((y1 + start, y1 + end))
            start = None

    if start is not None:
        end = len(supported_rows) - 1
        if end - start + 1 >= MIN_BAR_BAND_HEIGHT:
            bands.append((y1 + start, y1 + end))

    merged = []
    for band in bands:
        if not merged:
            merged.append(list(band))
        else:
            if band[0] - merged[-1][1] <= 4:
                merged[-1][1] = band[1]
            else:
                merged.append(list(band))

    merged = [(int(a), int(b)) for a, b in merged]
    return len(merged), merged


def count_vertical_bar_support(line, vertical_mask):
    h, w = vertical_mask.shape[:2]
    y = line["y1"]
    x1 = max(0, line["x1"])
    x2 = min(w - 1, line["x2"])

    scan_y1 = max(0, y - UP_SCAN_GAP - UP_SCAN_HEIGHT)
    scan_y2 = max(0, y - 1)

    if scan_y2 <= scan_y1 or x2 <= x1:
        return 0, []

    region = vertical_mask[scan_y1:scan_y2 + 1, x1:x2 + 1]
    col_counts = np.count_nonzero(region, axis=0)
    supported_cols = col_counts >= MIN_UP_RUN_PIXELS

    bands = []
    start = None

    for idx, value in enumerate(supported_cols):
        if value and start is None:
            start = idx
        elif not value and start is not None:
            end = idx - 1
            if end - start + 1 >= MIN_BAR_BAND_WIDTH:
                bands.append((x1 + start, x1 + end))
            start = None

    if start is not None:
        end = len(supported_cols) - 1
        if end - start + 1 >= MIN_BAR_BAND_WIDTH:
            bands.append((x1 + start, x1 + end))

    merged = []
    for band in bands:
        if not merged:
            merged.append(list(band))
        else:
            if band[0] - merged[-1][1] <= 4:
                merged[-1][1] = band[1]
            else:
                merged.append(list(band))

    merged = [(int(a), int(b)) for a, b in merged]
    return len(merged), merged


def find_left_axes_by_support_scanning(vertical_lines, horizontal_mask):
    axes = []

    for line in vertical_lines:
        support_count, bands = count_horizontal_bar_support(line, horizontal_mask)

        if support_count < MIN_SUPPORTING_BARS:
            continue

        y1 = min(b[0] for b in bands)
        y2 = max(b[1] for b in bands)
        x = line["x1"]

        pad = int((y2 - y1) * 0.12)
        y1 = max(0, y1 - pad)
        y2 = min(horizontal_mask.shape[0] - 1, y2 + pad)

        score = support_count * 120 + (y2 - y1) + line["height"] * 0.25

        axes.append({
            "source": "opencv",
            "axis_type": "left_axis",
            "x1": int(x),
            "y1": int(y1),
            "x2": int(x),
            "y2": int(y2),
            "supporting_bars": int(support_count),
            "score": float(score)
        })

    return axes


def find_bottom_axes_by_support_scanning(horizontal_lines, vertical_mask):
    axes = []

    for line in horizontal_lines:
        support_count, bands = count_vertical_bar_support(line, vertical_mask)

        if support_count < MIN_SUPPORTING_BARS:
            continue

        x1 = min(b[0] for b in bands)
        x2 = max(b[1] for b in bands)
        y = line["y1"]

        pad = int((x2 - x1) * 0.12)
        x1 = max(0, x1 - pad)
        x2 = min(vertical_mask.shape[1] - 1, x2 + pad)

        score = support_count * 120 + (x2 - x1) + line["width"] * 0.25

        axes.append({
            "source": "opencv",
            "axis_type": "bottom_x_axis",
            "x1": int(x1),
            "y1": int(y),
            "x2": int(x2),
            "y2": int(y),
            "supporting_bars": int(support_count),
            "score": float(score)
        })

    return axes


def remove_duplicate_axes(axes, tolerance=10):
    if not axes:
        return []

    axes = sorted(axes, key=lambda a: a["score"], reverse=True)
    kept = []

    for axis in axes:
        duplicate = False

        for other in kept:
            same_type = axis["axis_type"] == other["axis_type"]

            if not same_type:
                continue

            if axis["axis_type"] == "left_axis":
                close_x = abs(axis["x1"] - other["x1"]) <= tolerance
                overlap_y = min(axis["y2"], other["y2"]) - max(axis["y1"], other["y1"])
                if close_x and overlap_y > 0:
                    duplicate = True
                    break

            if axis["axis_type"] == "bottom_x_axis":
                close_y = abs(axis["y1"] - other["y1"]) <= tolerance
                overlap_x = min(axis["x2"], other["x2"]) - max(axis["x1"], other["x1"])
                if close_y and overlap_x > 0:
                    duplicate = True
                    break

        if not duplicate:
            kept.append(axis)

    return kept


def run_opencv_axis_detection(image):
    dark_mask = create_dark_mask(image)
    cleaned_mask = remove_outer_border(dark_mask)

    horizontal_structure_mask = make_horizontal_structure_mask(cleaned_mask)
    vertical_structure_mask = make_vertical_structure_mask(cleaned_mask)

    vertical_lines, vertical_lines_mask = detect_vertical_line_candidates(cleaned_mask, image.shape)
    horizontal_lines, horizontal_lines_mask = detect_horizontal_line_candidates(cleaned_mask, image.shape)

    left_axes = find_left_axes_by_support_scanning(vertical_lines, horizontal_structure_mask)
    bottom_axes = find_bottom_axes_by_support_scanning(horizontal_lines, vertical_structure_mask)

    axes = remove_duplicate_axes(left_axes + bottom_axes)
    axes = sorted(axes, key=lambda a: a["score"], reverse=True)

    return {
        "dark_mask": dark_mask,
        "cleaned_mask": cleaned_mask,
        "horizontal_structure_mask": horizontal_structure_mask,
        "vertical_structure_mask": vertical_structure_mask,
        "vertical_lines_mask": vertical_lines_mask,
        "horizontal_lines_mask": horizontal_lines_mask,
        "axes": axes
    }


def draw_axes(image, axes):
    output = image.copy()

    for i, axis in enumerate(axes[:MAX_AXES_TO_DRAW], start=1):
        x1, y1, x2, y2 = axis["x1"], axis["y1"], axis["x2"], axis["y2"]

        if axis["axis_type"] == "left_axis":
            color = (255, 0, 0)
            label = f"L{i} bars={axis['supporting_bars']}"
        else:
            color = (0, 180, 0)
            label = f"B{i} bars={axis['supporting_bars']}"

        cv2.line(output, (x1, y1), (x2, y2), color, 3)
        cv2.putText(
            output,
            label,
            (x1 + 5, max(15, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            color,
            2
        )

    return output


def axes_to_dataframe(axes):
    rows = []

    for i, a in enumerate(axes, start=1):
        rows.append({
            "rank": i,
            "source": a.get("source"),
            "axis_type": a.get("axis_type"),
            "x1": a.get("x1"),
            "y1": a.get("y1"),
            "x2": a.get("x2"),
            "y2": a.get("y2"),
            "supporting_bars": a.get("supporting_bars"),
            "score": round(a.get("score", 0), 1)
        })

    return pd.DataFrame(rows)


## 3. Run OpenCV detection

In [ ]:
image = load_image(IMAGE_PATH)
show_image("Original image", image)

opencv_result = run_opencv_axis_detection(image)

show_image("Dark mask", opencv_result["dark_mask"])
show_image("Horizontal structure mask", opencv_result["horizontal_structure_mask"])
show_image("Vertical line candidates mask", opencv_result["vertical_lines_mask"])

axes_df = axes_to_dataframe(opencv_result["axes"])
display(axes_df)

opencv_output = draw_axes(image, opencv_result["axes"])
show_image("OpenCV axis detection result", opencv_output)

opencv_output_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_opencv_axes.png"
cv2.imwrite(str(opencv_output_path), opencv_output)
print("Saved OpenCV result to:", opencv_output_path)

# Part B - VLM validation

Dit deel is optioneel.

De VLM krijgt:
1. de originele afbeelding;
2. de OpenCV-output met getekende assen;
3. de tabel met coördinaten.

De VLM geeft daarna een beoordeling:

```text
- Which detections look correct?
- Which detections look like dividers/borders?
- Is the image mainly horizontal bar charts, vertical bar charts, or mixed?
- Are there missing axes?
```

## Belangrijk
Deze notebook gebruikt geen VLM automatisch tenzij jij zelf een API of lokaal model toevoegt.

De makkelijkste manier:
- run OpenCV;
- bekijk de output;
- gebruik de prompt hieronder in ChatGPT / een VLM;
- plak eventueel de output terug in je verslag.


In [ ]:
def create_vlm_prompt(axes_dataframe):
    prompt = f'''
You are validating axis detection results for bar chart images.

The OpenCV model detects two types of axes:
- left_axis: vertical axis on the left side of a horizontal bar chart
- bottom_x_axis: horizontal x-axis below a vertical bar chart

Task:
1. Look at the original chart image.
2. Look at the OpenCV output image with drawn detections.
3. Decide which detections are logical and which are likely false positives.
4. Pay attention to dividers, borders, table lines and labels.
5. Explain if the image contains horizontal bar charts, vertical bar charts, or both.
6. Mention if important axes are missing.

OpenCV detections:
{axes_dataframe.to_string(index=False)}

Return your answer in this structure:

Chart type:
Detected axes that look correct:
Detected axes that look wrong:
Missing axes:
Reasoning:
Final judgement:
'''
    return prompt

vlm_prompt = create_vlm_prompt(axes_df)
print(vlm_prompt)

## Optional: VLM API cell

Gebruik deze cel alleen als je een API key hebt en weet welk vision model je wil gebruiken.

Standaard staat deze cel uit/commented, zodat de notebook gewoon werkt zonder API.


In [ ]:
# OPTIONAL VLM API EXAMPLE
# This is intentionally commented out.
# Fill in your own model name and API key if you want to use it.
#
# %pip install openai
# from openai import OpenAI
#
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
#
# original_b64 = encode_image_to_base64(IMAGE_PATH)
# opencv_b64 = encode_image_to_base64(opencv_output_path)
#
# response = client.responses.create(
#     model="YOUR_VISION_MODEL_NAME",
#     input=[
#         {
#             "role": "user",
#             "content": [
#                 {"type": "input_text", "text": vlm_prompt},
#                 {"type": "input_image", "image_url": f"data:image/png;base64,{original_b64}"},
#                 {"type": "input_image", "image_url": f"data:image/png;base64,{opencv_b64}"}
#             ]
#         }
#     ]
# )
#
# print(response.output_text)


## Optional: Manual VLM workflow

Als je geen API wil gebruiken, doe dit:

1. Run deze notebook tot en met de OpenCV-output.
2. Open de originele afbeelding.
3. Open de output-afbeelding uit:

```text
output/vlm_opencv/
```

4. Kopieer de VLM-prompt uit de vorige cel.
5. Plak de prompt + afbeeldingen in een VLM zoals ChatGPT Vision.
6. Gebruik het antwoord als kwalitatieve validatie in je verslag.


# Part C - Batch mode for OpenCV outputs

Deze cel maakt OpenCV-outputafbeeldingen voor alle afbeeldingen in je dataset.  
Die kun je daarna gebruiken voor VLM-validatie of handmatige evaluatie.


In [ ]:
DATASET_DIR = Path("../Dataset")
BATCH_OUTPUT_DIR = OUTPUT_DIR / "batch"
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_extensions = [".png", ".jpg", ".jpeg", ".webp"]
image_paths = []

for ext in image_extensions:
    image_paths.extend(DATASET_DIR.rglob(f"*{ext}"))

print("Found images:", len(image_paths))

summary_rows = []

for image_path in image_paths:
    try:
        img = load_image(image_path)
        res = run_opencv_axis_detection(img)
        output_img = draw_axes(img, res["axes"])

        relative = image_path.relative_to(DATASET_DIR)
        save_path = BATCH_OUTPUT_DIR / relative
        save_path.parent.mkdir(parents=True, exist_ok=True)

        cv2.imwrite(str(save_path), output_img)

        summary_rows.append({
            "image": str(relative),
            "axes_detected": len(res["axes"]),
            "left_axes": sum(1 for a in res["axes"] if a["axis_type"] == "left_axis"),
            "bottom_x_axes": sum(1 for a in res["axes"] if a["axis_type"] == "bottom_x_axis"),
            "output": str(save_path)
        })

    except Exception as e:
        summary_rows.append({
            "image": str(image_path),
            "error": str(e)
        })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "opencv_batch_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df.head())
print("Saved summary to:", summary_path)
print("Saved batch outputs to:", BATCH_OUTPUT_DIR)

# Part D - Text for report

Je kunt dit gebruiken in je verslag:

> This project uses a hybrid computer vision and VLM validation approach. OpenCV is used to detect possible axes with pixel-based line-support scanning. For horizontal bar charts, vertical line candidates are scanned on the right side to check whether multiple horizontal bar segments start near the line. For vertical bar charts, horizontal line candidates are scanned above the line to check whether multiple vertical bar segments end near the line. This produces candidate coordinates for `left_axis` and `bottom_x_axis`. Because OpenCV is rule-based and can confuse axes with dividers or borders, a Vision-Language Model can be used as an additional validation layer. The VLM receives the original image, the OpenCV output image and the detected coordinates, and then judges whether the detections are logical in the chart context.

Limitatie:

> The VLM is used for qualitative validation and reasoning, not for final pixel-perfect coordinate generation. The exact coordinates are produced by OpenCV. This is because VLMs can understand visual context, but they are usually less stable for precise line-coordinate extraction than computer vision methods.
